# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# =========================
# 1. Connect to internship repo
# =========================

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            "requirements.txt"
        ],
        check=True
    )
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# =========================
# 2. Hugging Face access
# =========================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

print("HF_TOKEN found.")

# =========================
# 3. DuckDB
# =========================

import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face.")

# =========================
# 4. March 2026 warehouse
# =========================

march_df = con.sql("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        sessions_ai,
        sessions_paid,
        sessions_direct,
        sessions_social,
        sessions_organic,
        month
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Working directory:", os.getcwd())
print("March 2026 shape:", march_df.shape)
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

display(march_df.head())

HF_TOKEN found.
DuckDB connected to Hugging Face.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
March 2026 shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,sessions_paid,sessions_direct,sessions_social,sessions_organic,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

One row represents one observed report date for one pseudonymized client and one pseudonymized content item.

For this contract, I use the March 2026 slice of the daily performance table.

The analysis window is 2026-03-01 through 2026-03-31.

The output is intended for decision-support and ranking, not for identifying clients or predicting an outcome from future information.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify grain and date window

grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(
            DISTINCT CONCAT(
                client_hash_id,
                '||',
                content_hash_id,
                '||',
                CAST(report_date AS VARCHAR)
            )
        ) AS unique_client_content_date_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_clients,unique_content_items,unique_client_content_date_rows
0,9841378,55,331437,9841378


In [20]:
date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(date_check)

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

- gsc_impressions — observed Google Search impressions.
- gsc_clicks — observed Google Search clicks.
- gsc_sum_position — observed aggregate search-position signal.
- scroll_events — observed engagement events.
- sessions_organic — observed organic-session signal.

### Label / proxy

For this contract, declining performance is treated as a proxy decision target from the earlier refresh lane framing.

The label/proxy is kept conceptually separate from the feature fields so that label-derived information is not used as an input.

### Context

- report_date — observation date.
- client_has_gsc — whether GSC exists for the client.
- client_has_ga4 — whether GA4 exists for the client.
- gsc_data_available — whether GSC data is available for the row.
- ga4_data_available — whether GA4 data is available for the row.
- month — warehouse partition/month.

### Excluded

- Future-window outcome fields — excluded because they would not be known at decision time.
- Trend/label-derived fields — excluded because they can leak the target into the features.
- Client-identifying information — excluded because the task is intended to remain pseudonymized.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verification query 1: grain

q1 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT CONCAT(
                client_hash_id,
                '||',
                content_hash_id,
                '||',
                CAST(report_date AS VARCHAR)
            )
        ) AS unique_client_content_date_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date_rows
0,9841378,9841378


In [23]:
# Verification query 2: row count and date span

q2 = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(q2)

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [24]:
# Verification query 3: availability

q3 = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

display(q3)

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This warehouse is not a perfectly balanced panel. Different clients have different history depths, so absence of a row should not automatically be interpreted as zero activity.

GSC and GA4 availability also differ across observations. A missing value can therefore represent unavailable source data rather than a true zero.

The daily table provides observed performance for the available reporting window, but it cannot by itself establish causality or guarantee that a content refresh will improve future performance.

The March slice is used as a development slice. Future outcomes should not be used as features when constructing a decision-time baseline.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.